In [ ]:
import re, glob, os, pandas as pd

def collect(kind: str):
    # matches: features_chunk_<num>_<KIND>_...bins.csv
    rx = re.compile(rf"^features_chunk_(\d+)_{kind}_.*?bins\.csv$")
    files = [
        f for f in glob.glob(f"features_chunk_*_{kind}_*bins.csv")
        if rx.search(os.path.basename(f))
    ]
    files.sort(key=lambda s: int(rx.search(os.path.basename(s)).group(1)))
    if not files:
        raise FileNotFoundError(f"No {kind} chunk files matched.")
    return files

def merge(kind: str, outname: str):
    files = collect(kind)
    cols = None
    parts = []
    for f in files:
        df = pd.read_csv(f)
        if cols is None:
            cols = list(df.columns)
        elif list(df.columns) != cols:
            raise ValueError(f"Column mismatch in {f}")
        parts.append(df)
    out = pd.concat(parts, ignore_index=True)
    out.to_csv(outname, index=False)
    print(f"Wrote {outname} with {len(out):,} rows from {len(files)} chunks.")

merge("TSS", "merged_TSS_500bp_20bins.csv")
merge("TTS", "merged_TTS_500bp_20bins.csv")